<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/bank_metrics_fairness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bank metrics and fairness

PR-AUC and F2 summarise a model well, but a bank asks different questions: *if the bank's analysts can only review a fixed number of transactions, how much fraud do we catch, and how many dollars?* And: *does the model miss more fraud for one group of customers than another?*

In this notebook we answer both, using the same protocol as `time_honest_evaluation.ipynb` (time-based validation, Random Forest Tuned, five seeds, 95% day-block bootstrap intervals). We do three things:

1. **Out-of-fold `category_risk`.** Until now, each training transaction's `category_risk` was calculated from data that included its own fraud label. We now calculate it out-of-fold, so no training row helps build its own encoding.
2. **Alert budgets.** Recall, dollar recall and precision when analysts review only the top 0.5%, 1% or 2% of transactions.
3. **Fairness.** We retrain the model without gender, and without both gender and age, and compare the overall performance and the share of fraud missed for each group.

## 1. Environment setup
The same setup as our earlier notebooks. We type credentials in at runtime, so they are never saved in the notebook.

In [4]:
# 1.1 Download the Sparkov dataset from Kaggle
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

!kaggle datasets download -d kartik2112/fraud-detection -p /content/data/sparkov --unzip

Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
100% 202M/202M [00:02<00:00, 95.4MB/s]



In [5]:
# 1.2 Clone the project repository
github_token = getpass('Paste your GitHub token: ')
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'
!git clone {repo_url}

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 205, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 205 (delta 49), reused 42 (delta 42), pack-reused 149 (from 1)
Receiving objects: 100% (205/205), 11.55 MiB | 15.85 MiB/s, done.
Resolving deltas: 100% (127/127), done.


In [6]:
# 1.3 Move into the repository and place the data in data/
%cd Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
!mv /content/data/sparkov data/

/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


### 1.4 Saving progress to Google Drive
We save each finished model and seed to Google Drive straight away, so if Colab disconnects, re-running the notebook skips everything already finished.

Our reference model (Random Forest Tuned with the earlier `category_risk`) was already trained in `time_honest_evaluation.ipynb`, so we load its saved predictions from that notebook's folder instead of training it again.

In [8]:
# 1.4 Connect Google Drive for saving progress
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/fraud_checkpoints/bank_metrics_fairness'
REFERENCE_DIR  = '/content/drive/MyDrive/fraud_checkpoints/time_honest_evaluation'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('checkpoints ->', CHECKPOINT_DIR)
print('reference model loaded from ->', REFERENCE_DIR)

Mounted at /content/drive
checkpoints -> /content/drive/MyDrive/fraud_checkpoints/bank_metrics_fairness
reference model loaded from -> /content/drive/MyDrive/fraud_checkpoints/time_honest_evaluation


In [9]:
# 1.5 Imports and project helpers
import sys
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import average_precision_score

sys.path.append(os.getcwd())
from src.split import split_sparkov
from src.utils import fit_category_risk, apply_category_risk, add_prev_txn_features
from src.evaluation import (chronological_val_split, run_seeds, bootstrap_model,
                            summarise, paired_difference, format_ci)

## 2. Features
The same six features as in our earlier notebooks. We also keep the transaction amount in dollars (for dollar recall), gender, and an age band for the fairness checks.

In [10]:
# 2.1 One timeline for train + test, with a fresh index
sparkov_train, sparkov_test = split_sparkov()
sparkov_train['_split'] = 'train'
sparkov_test['_split']  = 'test'
full_raw = (pd.concat([sparkov_train, sparkov_test])
            .sort_values('trans_date_trans_time')
            .reset_index(drop=True))
TIME = 'trans_date_trans_time'
print(full_raw.shape)

Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34
(1852394, 24)


In [11]:
# 2.2 Build the features (same definitions as before), plus an age band for the fairness checks
def add_features(df):
    df = add_prev_txn_features(df, 'cc_num', TIME, 'merch_lat', 'merch_long')
    ts = pd.to_datetime(df[TIME])
    df['velocity_log'] = np.log1p(df['implied_velocity_kmh'].fillna(0))
    df['is_night']     = ts.dt.hour.isin([22, 23, 0, 1, 2, 3]).astype(int)
    df['amt_log']      = np.log1p(df['amt'])
    df['age']          = (ts - pd.to_datetime(df['dob'])).dt.days / 365.25
    df['gender_flag']  = (df['gender'] == 'M').astype(int)
    df['day']          = ts.dt.floor('D')   # used only by the bootstrap
    df['age_band']     = pd.cut(df['age'], bins=[0, 30, 45, 60, 200], right=False,
                                labels=['under 30', '30-44', '45-59', '60+']).astype(str)
    return df

full = add_features(full_raw)

## 3. Time-based split and `category_risk`
### 3.1 Split
The same split as `time_honest_evaluation.ipynb`: the last 20% of the training period is validation.

We first build `category_risk` the way we did before: fitted on the whole training part, then applied to every row. We keep this version for the reference model.

In [12]:
# 3.1 Split by time and build category_risk the earlier way (fitted on the whole training part)
train = full[full['_split'] == 'train']
test  = full[full['_split'] == 'test'].copy()
train_sub, val, cutoff = chronological_val_split(train, TIME, val_frac=0.2)

risk_map = fit_category_risk(train_sub, 'category', 'is_fraud')
for df in (train_sub, val, test):
    df['category_risk'] = apply_category_risk(df, 'category', risk_map)

y_tr, y_val, y_test = train_sub['is_fraud'], val['is_fraud'], test['is_fraud']
test_days = test['day'].values

print('cutoff:', cutoff)
for name, df in [('train_sub', train_sub), ('val', val), ('test', test)]:
    print(f"{name:9s} rows={len(df):>9,}  fraud rate={df['is_fraud'].mean():.4%}")

cutoff: 2020-03-06 00:10:57.200000
train_sub rows=1,036,901  fraud rate=0.5756%
val       rows=  259,774  fraud rate=0.5921%
test      rows=  555,719  fraud rate=0.3860%


### 3.2 Out-of-fold `category_risk`
With the earlier version, each training row's `category_risk` includes that row's own fraud label. The model can then learn a little from its own answers.

Out-of-fold encoding avoids this. We split the training part into five folds. For each fold, we calculate the category fraud rates from the other four folds and use them to encode that fold. Every training row is then encoded only from rows other than itself.

Validation and test keep the rates fitted on the whole training part, because their labels were never used. The cell prints how much the encoding changes for the training rows.

In [13]:
# 3.2 Out-of-fold category_risk for the training part
def out_of_fold_category_risk(train_df, n_folds=5, seed=0):
    encoded = np.zeros(len(train_df))
    folds = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for fit_rows, encode_rows in folds.split(train_df):
        fold_map = fit_category_risk(train_df.iloc[fit_rows], 'category', 'is_fraud')
        encoded[encode_rows] = apply_category_risk(train_df.iloc[encode_rows], 'category', fold_map).values
    return encoded

train_sub['category_risk_oof'] = out_of_fold_category_risk(train_sub)
val['category_risk_oof']  = val['category_risk']    # validation and test are unchanged
test['category_risk_oof'] = test['category_risk']

diff = (train_sub['category_risk_oof'] - train_sub['category_risk']).abs()
print(f"training rows: mean change {diff.mean():.6f}, largest change {diff.max():.6f}")

training rows: mean change 0.000126, largest change 0.000598


## 4. Models
### 4.1 Runs
| Run | `category_risk` | Features |
|---|---|---|
| Reference | earlier version | the six features (loaded from `time_honest_evaluation.ipynb`) |
| Six, out-of-fold | out-of-fold | the six features |
| No gender | out-of-fold | the six features without `gender_flag` |
| No gender, no age | out-of-fold | the six features without `gender_flag` and `age` |

All runs use the Random Forest Tuned settings, with five seeds each.

In [14]:
# 4.1 Feature sets and the model
SIX = ['amt_log', 'category_risk', 'is_night', 'velocity_log', 'age', 'gender_flag']
OOF_SIX = ['amt_log', 'category_risk_oof', 'is_night', 'velocity_log', 'age', 'gender_flag']
FEATURE_SETS = {
    'Six, out-of-fold':  OOF_SIX,
    'No gender':         [f for f in OOF_SIX if f != 'gender_flag'],
    'No gender, no age': [f for f in OOF_SIX if f not in ('gender_flag', 'age')],
}

def make_rf(n_features):
    # Random Forest Tuned, with max_features capped at the number of features
    return lambda s: RandomForestClassifier(n_estimators=200, max_features=min(3, n_features),
                                            min_samples_leaf=5, random_state=s, n_jobs=-1)

SEEDS = [42, 43, 44, 45, 46]
REFERENCE = 'Reference'

### 4.2 Reference model
We load the five saved Random Forest Tuned runs from `time_honest_evaluation.ipynb`. If they are not found, the same model is trained here instead. As a check, the loaded predictions must give the same test PR-AUC as before (about 0.83 for each seed); if the rows were in a different order, it would drop to about 0.004.

In [15]:
# 4.2 Load the reference model's saved predictions
runs = {}
runs[REFERENCE] = run_seeds('Random Forest Tuned', make_rf(len(SIX)),
                            train_sub[SIX], y_tr, val[SIX], y_val, test[SIX], SEEDS,
                            checkpoint_dir=REFERENCE_DIR)

for r in runs[REFERENCE]:
    pr = average_precision_score(y_test, r['test_proba'])
    print(f"seed {r['seed']}: test PR-AUC {pr:.4f}")
    assert pr > 0.8, 'loaded predictions do not line up with our test rows'

Random Forest Tuned seed 42: loaded from checkpoint
Random Forest Tuned seed 43: loaded from checkpoint
Random Forest Tuned seed 44: loaded from checkpoint
Random Forest Tuned seed 45: loaded from checkpoint
Random Forest Tuned seed 46: loaded from checkpoint
seed 42: test PR-AUC 0.8337
seed 43: test PR-AUC 0.8341
seed 44: test PR-AUC 0.8341
seed 45: test PR-AUC 0.8341
seed 46: test PR-AUC 0.8346


### 4.3 Training the new runs
We train the three new runs with five seeds each. This is the longest step in the notebook.

In [16]:
# 4.3 Train every run with every seed (finished runs are loaded from Drive)
for name, cols in FEATURE_SETS.items():
    runs[name] = run_seeds(name, make_rf(len(cols)),
                           train_sub[cols], y_tr, val[cols], y_val, test[cols], SEEDS,
                           checkpoint_dir=CHECKPOINT_DIR)

Six, out-of-fold seed 42: loaded from checkpoint
Six, out-of-fold seed 43: loaded from checkpoint
Six, out-of-fold seed 44: loaded from checkpoint
Six, out-of-fold seed 45: loaded from checkpoint
Six, out-of-fold seed 46: loaded from checkpoint
No gender seed 42: loaded from checkpoint
No gender seed 43: loaded from checkpoint
No gender seed 44: loaded from checkpoint
No gender seed 45: loaded from checkpoint
No gender seed 46: loaded from checkpoint
No gender, no age seed 42: loaded from checkpoint
No gender, no age seed 43: loaded from checkpoint
No gender, no age seed 44: loaded from checkpoint
No gender, no age seed 45: loaded from checkpoint
No gender, no age seed 46: loaded from checkpoint


## 5. Overall performance
### 5.1 Results table
As in our earlier notebooks: 500 day-block resamples, the same `BOOT_SEED` for every run, seed average with a 95% interval.

In [17]:
# 5.1 Bootstrap each run and build the results table
N_BOOT = 500
BOOT_SEED = 2026

boots, summaries = {}, {}
for name, seed_runs in runs.items():
    pairs = [(r['test_proba'], r['threshold']) for r in seed_runs]
    boots[name] = bootstrap_model(y_test, pairs, test_days, n_boot=N_BOOT, boot_seed=BOOT_SEED)
    summaries[name] = summarise(y_test, pairs, boots[name])
    summaries[name]['Val PR-AUC'] = np.mean([r['val_pr_auc'] for r in seed_runs])
    print(f"{name}: done")

summary_df = pd.DataFrame(summaries).T
table = pd.DataFrame({
    'Val PR-AUC': summary_df['Val PR-AUC'].astype(float).round(4),
    'PR-AUC [95% CI]': [format_ci(r, 'PR-AUC') for _, r in summary_df.iterrows()],
    'F2 [95% CI]':  [format_ci(r, 'F2')  for _, r in summary_df.iterrows()],
    'MCC [95% CI]': [format_ci(r, 'MCC') for _, r in summary_df.iterrows()],
    'Thresholds (per seed)': summary_df['Thresholds'],
})
table

Reference: done
Six, out-of-fold: done
No gender: done
No gender, no age: done


,Val PR-AUC,PR-AUC [95% CI],F2 [95% CI],MCC [95% CI],Thresholds (per seed)
Reference,0.8769,"0.8341 [0.8140, 0.8529]","0.7892 [0.7689, 0.8056]","0.7180 [0.6941, 0.7385]","0.15, 0.16, 0.15, 0.19, 0.14"
"Six, out-of-fold",0.8540,"0.8076 [0.7848, 0.8293]","0.7633 [0.7420, 0.7824]","0.6955 [0.6694, 0.7168]","0.14, 0.15, 0.14, 0.14, 0.16"
No gender,0.8514,"0.7996 [0.7771, 0.8221]","0.7452 [0.7226, 0.7644]","0.6629 [0.6355, 0.6846]","0.10, 0.12, 0.13, 0.13, 0.14"
"No gender, no age",0.8388,"0.7878 [0.7663, 0.8092]","0.7441 [0.7228, 0.7640]","0.6759 [0.6498, 0.6980]","0.15, 0.17, 0.18, 0.16, 0.18"


### 5.2 Paired differences
We compare out-of-fold `category_risk` with the reference, and then each fairness run with the out-of-fold model. If the 95% interval includes zero, we cannot tell the two apart.

In [18]:
# 5.2 Paired differences (first run minus second run)
COMPARISONS = [('Six, out-of-fold', REFERENCE),
               ('No gender', 'Six, out-of-fold'),
               ('No gender, no age', 'Six, out-of-fold')]
diff_rows = []
for a, b in COMPARISONS:
    for metric in ['PR-AUC', 'F2', 'MCC']:
        d = paired_difference(boots[a], boots[b], metric)
        diff_rows.append({'Comparison': f'{a} minus {b}', 'Metric': metric,
                          **{k: round(v, 4) for k, v in d.items()}})
diff_df = pd.DataFrame(diff_rows).rename(columns={'P(A > B)': 'P(first > second)'})
diff_df

,Comparison,Metric,mean diff,CI low,CI high,P(first > second)
0,"Six, out-of-fold minus Reference",PR-AUC,-0.0265,-0.0318,-0.0216,0.0
1,"Six, out-of-fold minus Reference",F2,-0.0260,-0.0321,-0.0200,0.0
2,"Six, out-of-fold minus Reference",MCC,-0.0227,-0.0286,-0.0170,0.0
3,"No gender minus Six, out-of-fold",PR-AUC,-0.0079,-0.0121,-0.0038,0.0
4,"No gender minus Six, out-of-fold",F2,-0.0180,-0.0228,-0.0134,0.0
5,"No gender minus Six, out-of-fold",MCC,-0.0325,-0.0374,-0.0279,0.0
6,"No gender, no age minus Six, out-of-fold",PR-AUC,-0.0197,-0.0272,-0.0119,0.0
7,"No gender, no age minus Six, out-of-fold",F2,-0.0190,-0.0256,-0.0122,0.0
8,"No gender, no age minus Six, out-of-fold",MCC,-0.0193,-0.0259,-0.0124,0.0


## 6. Alert budgets
A bank's analysts can only review a limited number of alerts. We rank all test transactions by the model's score and flag the top 0.5%, 1% or 2%. For each budget we report:
- **Recall:** the share of fraud transactions that are flagged;
- **Dollar recall:** the share of fraud *dollars* that are flagged, which is what the bank can try to recover;
- **Precision:** the share of flagged transactions that are fraud.

The test period has a fraud rate of 0.386%, so every fraud fits inside even the 0.5% budget, and a perfect model would reach a recall of 1.0 at every budget. As the budget grows, more alerts are raised for the same number of frauds, so precision usually falls.

The intervals use the same day-block resamples as Section 5. For a resample, the budget is a share of the resampled transactions.

In [19]:
# 6.1 Test-period totals
y_arr   = y_test.values
amt_arr = test['amt'].values
print(f"test transactions: {len(y_arr):,}")
print(f"fraud transactions: {y_arr.sum():,}  ({y_arr.mean():.3%})")
print(f"fraud dollars: ${amt_arr[y_arr == 1].sum():,.0f}")
for b in [0.005, 0.01, 0.02]:
    print(f"budget {b:.1%}: {int(b * len(y_arr)):,} alerts")

test transactions: 555,719
fraud transactions: 2,145  (0.386%)
fraud dollars: $1,133,325
budget 0.5%: 2,778 alerts
budget 1.0%: 5,557 alerts
budget 2.0%: 11,114 alerts


In [20]:
# 6.2 Recall, dollar recall and precision at each budget
BUDGETS = [0.005, 0.01, 0.02]

def budget_scores(w, y, a, budget):
    # w, y, a: row weights, labels and amounts, sorted from the highest score to the lowest
    flagged = np.cumsum(w) <= budget * w.sum()
    caught = w * y * flagged
    return {'Recall': caught.sum() / (w * y).sum(),
            'Dollar recall': (caught * a).sum() / (w * y * a).sum(),
            'Precision': caught.sum() / (w * flagged).sum()}

def day_weight_draws(days, n_boot, boot_seed):
    # the same day resamples as bootstrap_model in src/evaluation.py
    days = pd.Series(days).reset_index(drop=True)
    unique_days = days.unique()
    rng = np.random.default_rng(boot_seed)
    for _ in range(n_boot):
        drawn = rng.choice(unique_days, size=len(unique_days), replace=True)
        yield days.map(pd.Series(drawn).value_counts()).fillna(0).values

# sort each seed's test rows once, from the highest score to the lowest
sorted_runs = {}
for name, seed_runs in runs.items():
    sorted_runs[name] = []
    for r in seed_runs:
        order = np.argsort(-r['test_proba'], kind='stable')
        sorted_runs[name].append((order, y_arr[order], amt_arr[order]))

def run_budget_scores(name, weights, budget):
    # average over the five seeds
    scores = [budget_scores(weights[o], y_s, a_s, budget) for o, y_s, a_s in sorted_runs[name]]
    return pd.DataFrame(scores).mean()

# point estimates on the full test set
ones = np.ones(len(y_arr))
point = {(n, b): run_budget_scores(n, ones, b) for n in runs for b in BUDGETS}

# bootstrap
boot_rows = []
for w in day_weight_draws(test_days, N_BOOT, BOOT_SEED):
    for n in runs:
        for b in BUDGETS:
            boot_rows.append({'Run': n, 'Budget': b, **run_budget_scores(n, w, b)})
boot_budget = pd.DataFrame(boot_rows)

budget_rows = []
for (n, b), p in point.items():
    bb = boot_budget[(boot_budget['Run'] == n) & (boot_budget['Budget'] == b)]
    row = {'Run': n, 'Budget': f'{b:.1%}', 'Alerts': int(b * len(y_arr))}
    for m in ['Recall', 'Dollar recall', 'Precision']:
        lo, hi = np.percentile(bb[m], [2.5, 97.5])
        row[f'{m} [95% CI]'] = f"{p[m]:.4f} [{lo:.4f}, {hi:.4f}]"
    budget_rows.append(row)
budget_df = pd.DataFrame(budget_rows)
budget_df

,Run,Budget,Alerts,Recall [95% CI],Dollar recall [95% CI],Precision [95% CI]
0,Reference,0.5%,2778,"0.8371 [0.8107, 0.8575]","0.9452 [0.9286, 0.9597]","0.6464 [0.5760, 0.7084]"
1,Reference,1.0%,5557,"0.9171 [0.9030, 0.9296]","0.9817 [0.9756, 0.9869]","0.3540 [0.3120, 0.3947]"
2,Reference,2.0%,11114,"0.9490 [0.9384, 0.9584]","0.9916 [0.9876, 0.9946]","0.1832 [0.1615, 0.2048]"
3,"Six, out-of-fold",0.5%,2778,"0.8131 [0.7886, 0.8336]","0.9484 [0.9342, 0.9598]","0.6278 [0.5570, 0.6907]"
4,"Six, out-of-fold",1.0%,5557,"0.8906 [0.8754, 0.9045]","0.9798 [0.9740, 0.9850]","0.3438 [0.3034, 0.3813]"
5,"Six, out-of-fold",2.0%,11114,"0.9302 [0.9182, 0.9405]","0.9884 [0.9841, 0.9918]","0.1795 [0.1582, 0.2009]"
6,No gender,0.5%,2778,"0.8017 [0.7778, 0.8218]","0.9406 [0.9267, 0.9522]","0.6190 [0.5482, 0.6806]"
7,No gender,1.0%,5557,"0.8890 [0.8731, 0.9025]","0.9770 [0.9703, 0.9830]","0.3431 [0.3022, 0.3806]"
8,No gender,2.0%,11114,"0.9308 [0.9188, 0.9412]","0.9880 [0.9834, 0.9915]","0.1796 [0.1585, 0.2008]"
9,"No gender, no age",0.5%,2778,"0.7923 [0.7681, 0.8135]","0.9328 [0.9158, 0.9477]","0.6117 [0.5434, 0.6714]"


## 7. Fairness: missed fraud by group
For each group of customers, we calculate the **missed-fraud rate**: the share of that group's fraud transactions that the model does not flag at its F2 threshold. A fair model misses fraud at a similar rate for every group.

We look at gender (female and male) and four age bands. Only fraud transactions matter for this rate, so the calculation uses the fraud rows of the test period. The intervals use the same day-block resamples as before.

The gender gap is the male rate minus the female rate. The age gap is the band with the highest missed-fraud rate minus the band with the lowest; we choose these two bands on the full test set and keep them fixed in every resample.

In [21]:
# 7.1 Fraud transactions per group in the test period
fraud_rows = np.where(y_arr == 1)[0]
gender_f   = test['gender'].values[fraud_rows]
age_f      = test['age_band'].values[fraud_rows]
GENDERS    = ['F', 'M']
AGE_BANDS  = ['under 30', '30-44', '45-59', '60+']

print('fraud transactions by gender:', {g: int((gender_f == g).sum()) for g in GENDERS})
print('fraud transactions by age band:', {a: int((age_f == a).sum()) for a in AGE_BANDS})

fraud transactions by gender: {'F': 1164, 'M': 981}
fraud transactions by age band: {'under 30': 317, '30-44': 649, '45-59': 604, '60+': 575}


In [22]:
# 7.2 Missed-fraud rates and gaps
# for each seed: 1 if the model flags the fraud transaction at its F2 threshold, otherwise 0
flagged_f = {name: [(r['test_proba'][fraud_rows] >= r['threshold']).astype(int) for r in seed_runs]
             for name, seed_runs in runs.items()}

def missed_rates(name, w):
    # missed-fraud rate for each group, averaged over the five seeds; w = weights of the fraud rows
    out = {}
    for label, groups, values in [('Gender', gender_f, GENDERS), ('Age', age_f, AGE_BANDS)]:
        for v in values:
            in_group = (groups == v)
            out[v] = np.mean([np.sum(w * in_group * (1 - f)) / np.sum(w * in_group) for f in flagged_f[name]])
    out['Gender gap (M - F)'] = out['M'] - out['F']
    return out

point_fair = {n: missed_rates(n, np.ones(len(fraud_rows))) for n in runs}
boot_fair = pd.DataFrame([{'Run': n, **missed_rates(n, w[fraud_rows])}
                          for w in day_weight_draws(test_days, N_BOOT, BOOT_SEED) for n in runs])

# age gap: the age band with the highest missed-fraud rate minus the band with the lowest,
# both chosen on the full test set and then kept fixed in every resample
age_pairs = {}
for n in runs:
    rates = {a: point_fair[n][a] for a in AGE_BANDS}
    high, low = max(rates, key=rates.get), min(rates, key=rates.get)
    age_pairs[n] = f'{high} minus {low}'
    point_fair[n]['Age gap'] = rates[high] - rates[low]
    in_run = boot_fair['Run'] == n
    boot_fair.loc[in_run, 'Age gap'] = boot_fair.loc[in_run, high] - boot_fair.loc[in_run, low]

def fair_table(columns):
    rows = []
    for n in runs:
        bf = boot_fair[boot_fair['Run'] == n]
        row = {'Run': n}
        for c in columns:
            lo, hi = np.percentile(bf[c], [2.5, 97.5])
            row[c] = f"{point_fair[n][c]:.4f} [{lo:.4f}, {hi:.4f}]"
        rows.append(row)
    return pd.DataFrame(rows)

gender_df = fair_table(['F', 'M', 'Gender gap (M - F)'])
age_df    = fair_table(AGE_BANDS + ['Age gap'])
age_df['Age gap bands'] = age_df['Run'].map(age_pairs)
print('Missed-fraud rate by gender [95% CI]')
display(gender_df)
print('Missed-fraud rate by age band [95% CI]')
display(age_df)

Missed-fraud rate by gender [95% CI]


,Run,F,M,Gender gap (M - F)
0,Reference,"0.2069 [0.1803, 0.2345]","0.0734 [0.0585, 0.0893]","-0.1335 [-0.1644, -0.0984]"
1,"Six, out-of-fold","0.2419 [0.2119, 0.2707]","0.0968 [0.0814, 0.1157]","-0.1451 [-0.1775, -0.1075]"
2,No gender,"0.2337 [0.2050, 0.2620]","0.0987 [0.0832, 0.1174]","-0.1350 [-0.1690, -0.0990]"
3,"No gender, no age","0.2656 [0.2338, 0.2954]","0.1083 [0.0907, 0.1278]","-0.1574 [-0.1931, -0.1200]"


Missed-fraud rate by age band [95% CI]


,Run,under 30,30-44,45-59,60+,Age gap,Age gap bands
0,Reference,"0.1281 [0.0835, 0.1793]","0.2018 [0.1668, 0.2349]","0.1407 [0.1099, 0.1725]","0.0977 [0.0728, 0.1245]","0.1041 [0.0596, 0.1490]",30-44 minus 60+
1,"Six, out-of-fold","0.1451 [0.0998, 0.1951]","0.2419 [0.2024, 0.2772]","0.1659 [0.1353, 0.1982]","0.1277 [0.1009, 0.1587]","0.1143 [0.0655, 0.1664]",30-44 minus 60+
2,No gender,"0.1451 [0.1009, 0.1919]","0.2398 [0.2029, 0.2731]","0.1642 [0.1357, 0.1972]","0.1183 [0.0910, 0.1472]","0.1215 [0.0747, 0.1692]",30-44 minus 60+
3,"No gender, no age","0.1609 [0.1125, 0.2090]","0.2586 [0.2178, 0.2941]","0.1924 [0.1610, 0.2279]","0.1398 [0.1112, 0.1718]","0.1187 [0.0683, 0.1668]",30-44 minus 60+


## 8. Saving the results
We save every results table to `results/`, so the paper tables can be built from them.

In [23]:
# 8.1 Save the results tables
os.makedirs('results', exist_ok=True)
summary_df.to_csv('results/bank_fairness_summary.csv')
table.to_csv('results/bank_fairness_table.csv')
diff_df.to_csv('results/bank_fairness_paired_differences.csv', index=False)
budget_df.to_csv('results/bank_fairness_alert_budgets.csv', index=False)
gender_df.to_csv('results/bank_fairness_gender.csv', index=False)
age_df.to_csv('results/bank_fairness_age.csv', index=False)
print('saved to results/')

saved to results/


## 9. Discussion

**Out-of-fold `category_risk`.** The encoding itself barely changed. For the training rows, the mean change was 0.000126 and the largest was 0.000598. Even so, the test results fell compared with the reference: PR-AUC went from 0.8341 to 0.8076, a paired difference of -0.0265 with a 95% interval of [-0.0318, -0.0216]. F2 and MCC fell by a similar amount, and none of the intervals include zero. We did not test why. The earlier encoding may have leaked a very small amount, or the drop may be fold-to-fold noise. We did not run a check, so we do not claim that the reference was optimistic. We use the out-of-fold version for the fairness work because no training row helps encode itself.

**Alert budgets.** With the out-of-fold model, reviewing the top 1% of transactions (5,557 alerts) catches 89.1% of fraud transactions and 98.0% of fraud dollars, at 34.4% precision. At 0.5% (2,778 alerts) it catches 81.3% of fraud and 94.8% of fraud dollars, at 62.8% precision. At 2% (11,114 alerts) it catches 93.0% of fraud and 98.8% of fraud dollars, but precision falls to 18.0%. Dollar recall is higher than transaction recall at every budget, so the frauds the model misses tend to be the smaller ones. Going from 1% to 2% adds only about 4 points of recall for twice the review workload, so the gain slows down after 1%. The reference and out-of-fold intervals for dollar recall overlap at every budget (at 1%: 0.9817 [0.9756, 0.9869] against 0.9798 [0.9740, 0.9850]). We did not run a paired test on the budget metrics.

**Removing gender and age.** Removing gender lowered performance a little (paired differences against the out-of-fold model: PR-AUC -0.0079, F2 -0.0180, MCC -0.0325; all intervals exclude zero). Removing both gender and age lowered it somewhat more (PR-AUC -0.0197, F2 -0.0190, MCC -0.0193). At a fixed 1% budget the effect is small: dollar recall goes from 0.9798 to 0.9770 and 0.9736, and recall from 0.8906 to 0.8890 and 0.8669. Dropping these features costs little detection in this setup.

**Fairness.** The model misses much more fraud on women than on men. For the out-of-fold model the missed-fraud rate is 0.2419 for women and 0.0968 for men, a gap (M - F) of -0.1451 with a 95% interval of [-0.1775, -0.1075]. Removing gender left the gap about the same (-0.1350), and removing gender and age made it slightly wider (-0.1574). The intervals for the three runs overlap, so we see no evidence that dropping these features closes the gap. Missed-fraud rates for both genders rise only when gender and age are both removed (0.2656 for women, 0.1083 for men). For age, the 30-44 band is missed most often (0.2419) and the 60+ band least (0.1277), a gap of 0.1143 [0.0655, 0.1664] that stays about the same across runs (0.1215 and 0.1187). The gaps therefore do not seem to come from the gender and age columns alone. Amount, category and velocity may carry similar information, or fraud patterns may differ across groups. We did not test this.

**Limitations.**
- Sparkov's customers are synthetic, so the fairness results test our method, not the treatment of real people.
- Removing a feature does not remove information that other features carry about it, so a model without gender can still treat groups differently.
- The missed-fraud rate is one measure of fairness. We did not test others, such as the share of genuine customers wrongly flagged in each group.
- We did not run paired tests for the alert-budget metrics or the fairness gaps, so those comparisons rely on overlapping intervals.
- We did not test why out-of-fold encoding lowered performance. A noise check on the reference encoding would settle it.
- The under-30 band has only 317 fraud transactions, so its estimates are less precise.